# Visualize Rollout Dataset

This notebook loads a rollout dataset H5 file (generated by `collect_rollout_dataset.py`) and renders
a selected clip from each dataset for comparison:
- Original reference data
- Encoder-decoder rollouts
- Prior rollouts (logvar=-4, -2, 0, deterministic)

## Setup

In [ ]:
import os

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

import h5py
import jax
import jax.numpy as jnp
import mediapy as media
import mujoco
import numpy as np
from mujoco import mjx
from ml_collections import ConfigDict
from vnl_playground.tasks.rodent import imitation
from vnl_playground.tasks.rodent import wrappers as vnl_wrappers
from vnl_playground.tasks.rodent.reference_clips import ReferenceClips

print(f"JAX backend: {jax.default_backend()}")
print(f"JAX devices: {jax.devices()}")

## Configuration

Set the path to your rollout dataset H5 file and choose a clip index to visualize.

In [ ]:
# ============ CONFIGURATION ============
# Path to the rollout dataset H5 file
DATASET_PATH = "/home/mila/a/aidan.sirbu/track-mjx/data/discriminator_dataset_260114_162929_898338_23.h5"  # Update this path

# Path to reference clips (needed to create environment with correct model)
REFERENCE_DATA_PATH = (
    "/home/mila/a/aidan.sirbu/track-mjx/data/rodent/rodent_reference_clips.h5"
)

# Clip index to visualize (0-841)
CLIP_IDX = 100

# Rendering settings
RENDER_FPS = 50
CAMERA_NAME = "close_profile-rodent"
RENDER_HEIGHT = 480
RENDER_WIDTH = 640

## Load Dataset

In [ ]:
# Load the H5 file
with h5py.File(DATASET_PATH, "r") as f:
    print("Datasets in file:")
    for key in f.keys():
        print(f"  {key}: {f[key].shape}")
    print()
    print("Attributes:")
    for key, value in f.attrs.items():
        print(f"  {key}: {value}")

    # Load all datasets
    datasets = {}
    for key in f.keys():
        datasets[key] = f[key][:]

    num_clips = f.attrs["num_clips"]
    num_steps = f.attrs["num_steps"]

print(f"\nLoaded {len(datasets)} datasets")
print(f"Number of clips: {num_clips}")
print(f"Steps per clip: {num_steps}")

In [ ]:
# Dataset names for rendering (in display order)
DATASET_NAMES = [
    "original_qpos",
    "encoder_decoder_qpos",
    "prior_deterministic_qpos",
    "prior_logvar_-4_qpos",
    "prior_logvar_-2_qpos",
    "prior_logvar_0_qpos",
    "prior_predicted_logvar_qpos",
]

# Display names for each dataset
DISPLAY_NAMES = {
    "original_qpos": "Original (Reference)",
    "encoder_decoder_qpos": "Encoder-Decoder",
    "prior_deterministic_qpos": "Prior (Deterministic)",
    "prior_logvar_-4_qpos": "Prior (logvar=-4)",
    "prior_logvar_-2_qpos": "Prior (logvar=-2)",
    "prior_logvar_0_qpos": "Prior (logvar=0)",
    "prior_predicted_logvar_qpos": "Prior (Predicted logvar)",
}

# Verify all datasets exist
for name in DATASET_NAMES:
    if name in datasets:
        print(f"Found: {name}")
    else:
        print(f"MISSING: {name}")

## Create Environment for Rendering

We use the VNL imitation environment because it has the correct MuJoCo model with 74 qpos dimensions (7 root + 67 joints).

In [ ]:
# Create environment using defaults - only need to pass reference clips
# (default_config() provides all XML paths, timing, solver settings, etc.)
reference_clips = ReferenceClips(
    data_path=REFERENCE_DATA_PATH,
    n_frames_per_clip=250,
)

# Create environment - just pass the clips, everything else uses defaults
env = vnl_wrappers.FlattenObsWrapper(imitation.Imitation(clips=reference_clips))

print(f"Environment created")
print(f"  Model nq (qpos dim): {env.mj_model.nq}")
print(f"  Model nv (qvel dim): {env.mj_model.nv}")
print(f"\nAvailable cameras:")
for i in range(env.mj_model.ncam):
    print(f"  {i}: {env.mj_model.camera(i).name}")

## Rendering Functions

In [ ]:
def render_qpos_trajectory(
    qpos_trajectory: np.ndarray,
    env,
    camera_name: str = "close_profile-rodent",
    height: int = 480,
    width: int = 640,
) -> list[np.ndarray]:
    """Render a qpos trajectory to frames using the VNL environment.

    Args:
        qpos_trajectory: Array of shape (num_steps, qpos_dim).
        env: VNL environment with correct model.
        camera_name: Camera to render from.
        height: Frame height.
        width: Frame width.

    Returns:
        List of RGB frames.
    """
    mj_model = env.mj_model
    mj_data = mujoco.MjData(mj_model)
    renderer = mujoco.Renderer(mj_model, height=height, width=width)

    frames = []

    for qpos in qpos_trajectory:
        # Set qpos
        mj_data.qpos[:] = qpos

        # Forward kinematics
        mujoco.mj_forward(mj_model, mj_data)

        # Render
        renderer.update_scene(mj_data, camera=camera_name)
        frame = renderer.render()
        frames.append(frame.copy())

    renderer.close()
    return frames

## Select Clip and Render All Datasets

In [ ]:
# Validate clip index
assert 0 <= CLIP_IDX < num_clips, f"CLIP_IDX must be in range [0, {num_clips})"

print(f"Rendering clip {CLIP_IDX} from all datasets...")
print(f"Each clip has {num_steps} frames")
print()

In [ ]:
# Render all datasets for the selected clip
all_videos = {}

for dataset_name in DATASET_NAMES:
    if dataset_name not in datasets:
        print(f"Skipping {dataset_name} (not found)")
        continue

    display_name = DISPLAY_NAMES.get(dataset_name, dataset_name)
    print(f"Rendering {display_name}...")

    # Get qpos trajectory for this clip
    qpos_trajectory = datasets[dataset_name][CLIP_IDX]  # (num_steps, qpos_dim)

    # Render
    frames = render_qpos_trajectory(
        qpos_trajectory=qpos_trajectory,
        env=env,
        camera_name=CAMERA_NAME,
        height=RENDER_HEIGHT,
        width=RENDER_WIDTH,
    )

    all_videos[dataset_name] = frames
    print(f"  Rendered {len(frames)} frames")

print(f"\nRendered {len(all_videos)} videos")

## Display Videos

Each video shows the same clip index rendered from a different dataset.

In [ ]:
print(f"=" * 60)
print(f"CLIP {CLIP_IDX} - Original (Reference)")
print(f"=" * 60)
if "original_qpos" in all_videos:
    media.show_video(all_videos["original_qpos"], fps=RENDER_FPS)
else:
    print("Not available")

In [ ]:
print(f"=" * 60)
print(f"CLIP {CLIP_IDX} - Encoder-Decoder Rollout")
print(f"=" * 60)
if "encoder_decoder_qpos" in all_videos:
    media.show_video(all_videos["encoder_decoder_qpos"], fps=RENDER_FPS)
else:
    print("Not available")

In [ ]:
print(f"=" * 60)
print(f"CLIP {CLIP_IDX} - Prior (Deterministic)")
print(f"=" * 60)
if "prior_deterministic_qpos" in all_videos:
    media.show_video(all_videos["prior_deterministic_qpos"], fps=RENDER_FPS)
else:
    print("Not available")

In [ ]:
print(f"=" * 60)
print(f"CLIP {CLIP_IDX} - Prior (logvar=-4, std~0.14)")
print(f"=" * 60)
if "prior_logvar_-4_qpos" in all_videos:
    media.show_video(all_videos["prior_logvar_-4_qpos"], fps=RENDER_FPS)
else:
    print("Not available")

In [ ]:
print(f"=" * 60)
print(f"CLIP {CLIP_IDX} - Prior (logvar=-2, std~0.37)")
print(f"=" * 60)
if "prior_logvar_-2_qpos" in all_videos:
    media.show_video(all_videos["prior_logvar_-2_qpos"], fps=RENDER_FPS)
else:
    print("Not available")

In [ ]:
print(f"=" * 60)
print(f"CLIP {CLIP_IDX} - Prior (logvar=0, std=1.0)")
print(f"=" * 60)
if "prior_logvar_0_qpos" in all_videos:
    media.show_video(all_videos["prior_logvar_0_qpos"], fps=RENDER_FPS)
else:
    print("Not available")

## Quick Comparison Functions

Use these to quickly render and display different clips.

In [ ]:
def render_and_display_clip(clip_idx: int, dataset_name: str = "original_qpos"):
    """Quickly render and display a single clip from a dataset.

    Args:
        clip_idx: Clip index to render.
        dataset_name: Name of dataset to render from.
    """
    assert 0 <= clip_idx < num_clips, f"clip_idx must be in range [0, {num_clips})"
    assert dataset_name in datasets, f"Dataset {dataset_name} not found"

    display_name = DISPLAY_NAMES.get(dataset_name, dataset_name)
    print(f"Rendering clip {clip_idx} from {display_name}...")

    qpos_trajectory = datasets[dataset_name][clip_idx]
    frames = render_qpos_trajectory(
        qpos_trajectory=qpos_trajectory,
        env=env,
        camera_name=CAMERA_NAME,
        height=RENDER_HEIGHT,
        width=RENDER_WIDTH,
    )

    print(f"Clip {clip_idx} - {display_name}")
    media.show_video(frames, fps=RENDER_FPS)


def render_all_for_clip(clip_idx: int):
    """Render and display all datasets for a given clip.

    Args:
        clip_idx: Clip index to render.
    """
    assert 0 <= clip_idx < num_clips, f"clip_idx must be in range [0, {num_clips})"

    print(f"Rendering all datasets for clip {clip_idx}...\n")

    for dataset_name in DATASET_NAMES:
        if dataset_name not in datasets:
            continue

        display_name = DISPLAY_NAMES.get(dataset_name, dataset_name)
        qpos_trajectory = datasets[dataset_name][clip_idx]
        frames = render_qpos_trajectory(
            qpos_trajectory=qpos_trajectory,
            env=env,
            camera_name=CAMERA_NAME,
            height=RENDER_HEIGHT,
            width=RENDER_WIDTH,
        )

        print(f"=" * 40)
        print(f"Clip {clip_idx} - {display_name}")
        print(f"=" * 40)
        media.show_video(frames, fps=RENDER_FPS)

In [ ]:
# Example: render a specific clip from all datasets
render_all_for_clip(45)

In [ ]:
# Example: render a specific clip from a specific dataset
# render_and_display_clip(100, "encoder_decoder_qpos")

## Low-Pass Filter for Joint Angles

Apply a low-pass Butterworth filter to smooth joint angles (qpos indices 7 and onwards) while preserving the root position/orientation (indices 0-6).

In [ ]:
from scipy.signal import butter, filtfilt


def lowpass_filter_joint_angles(
    qpos_trajectory: np.ndarray,
    cutoff_freq: float = 5.0,
    sampling_freq: float = 50.0,
    order: int = 2,
) -> np.ndarray:
    """Apply a low-pass Butterworth filter to joint angles only.

    Filters qpos indices 7 and onwards (joint angles) while preserving
    indices 0-6 (root position and orientation) unchanged.

    Args:
        qpos_trajectory: Array of shape (num_steps, qpos_dim).
        cutoff_freq: Cutoff frequency in Hz (default 5 Hz).
        sampling_freq: Sampling frequency in Hz (default 50 Hz, matching RENDER_FPS).
        order: Filter order (default 2).

    Returns:
        Filtered qpos trajectory with same shape.
    """
    filtered = qpos_trajectory.copy()

    # Nyquist frequency
    nyquist = sampling_freq / 2.0
    normalized_cutoff = cutoff_freq / nyquist

    # Design Butterworth filter
    b, a = butter(order, normalized_cutoff, btype="low")

    # Apply filter to joint angles only (indices 7 onwards)
    joint_angles = qpos_trajectory[:, 7:]  # (num_steps, num_joints)

    # Filter each joint angle dimension independently
    for i in range(joint_angles.shape[1]):
        filtered[:, 7 + i] = filtfilt(b, a, joint_angles[:, i])

    return filtered

In [ ]:
# ============ LOW-PASS FILTER CONFIGURATION ============
# Dataset to filter
FILTER_DATASET = "prior_logvar_-2_qpos"  # Change this to filter a different dataset
FILTER_CLIP_IDX = 20  # Use same clip as above, or set a different one

# Filter parameters
CUTOFF_FREQ = 2.0  # Cutoff frequency in Hz (lower = smoother)
FILTER_ORDER = 2  # Filter order (higher = sharper cutoff)

print(f"Low-pass filter settings:")
print(f"  Dataset: {FILTER_DATASET}")
print(f"  Clip index: {FILTER_CLIP_IDX}")
print(f"  Cutoff frequency: {CUTOFF_FREQ} Hz")
print(f"  Sampling frequency: {RENDER_FPS} Hz")
print(f"  Filter order: {FILTER_ORDER}")

In [ ]:
# Get original qpos trajectory
assert FILTER_DATASET in datasets, f"Dataset {FILTER_DATASET} not found"
assert 0 <= FILTER_CLIP_IDX < num_clips, f"Clip index must be in range [0, {num_clips})"

original_qpos = datasets[FILTER_DATASET][FILTER_CLIP_IDX]

# Apply low-pass filter to joint angles
filtered_qpos = lowpass_filter_joint_angles(
    original_qpos,
    cutoff_freq=CUTOFF_FREQ,
    sampling_freq=RENDER_FPS,
    order=FILTER_ORDER,
)

print(f"Applied low-pass filter to joint angles (indices 7-{original_qpos.shape[1]-1})")
print(f"  Original shape: {original_qpos.shape}")
print(f"  Filtered shape: {filtered_qpos.shape}")

# Show difference statistics
diff = np.abs(filtered_qpos - original_qpos)
print(f"\nFiltering difference statistics (joint angles only):")
print(f"  Max absolute change: {diff[:, 7:].max():.6f}")
print(f"  Mean absolute change: {diff[:, 7:].mean():.6f}")
print(
    f"  Root (indices 0-6) unchanged: {np.allclose(original_qpos[:, :7], filtered_qpos[:, :7])}"
)

In [ ]:
# Render both original and filtered trajectories
display_name = DISPLAY_NAMES.get(FILTER_DATASET, FILTER_DATASET)

print(f"Rendering original (unfiltered) {display_name}...")
original_frames = render_qpos_trajectory(
    qpos_trajectory=original_qpos,
    env=env,
    camera_name=CAMERA_NAME,
    height=RENDER_HEIGHT,
    width=RENDER_WIDTH,
)
print(f"  Rendered {len(original_frames)} frames")

print(f"Rendering filtered {display_name}...")
filtered_frames = render_qpos_trajectory(
    qpos_trajectory=filtered_qpos,
    env=env,
    camera_name=CAMERA_NAME,
    height=RENDER_HEIGHT,
    width=RENDER_WIDTH,
)
print(f"  Rendered {len(filtered_frames)} frames")

In [ ]:
# Display original (unfiltered) video
print(f"=" * 60)
print(f"CLIP {FILTER_CLIP_IDX} - {display_name} (Original)")
print(f"=" * 60)
media.show_video(original_frames, fps=RENDER_FPS)

In [ ]:
# Display filtered video
print(f"=" * 60)
print(
    f"CLIP {FILTER_CLIP_IDX} - {display_name} (Low-Pass Filtered, cutoff={CUTOFF_FREQ}Hz)"
)
print(f"=" * 60)
media.show_video(filtered_frames, fps=RENDER_FPS)